# Notebook 26 — Joint FICO + DTI Manski Bounds

Extends FICO-only Manski bounds to include DTI. Addresses consensus criticism #2.

**Runtime:** ~10 minutes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
TABS = Path('../outputs/tables'); TABS.mkdir(exist_ok=True)
FIGS = Path('../outputs/figures'); FIGS.mkdir(exist_ok=True)

print('NB26: JOINT FICO + DTI MANSKI BOUNDS')
print('='*65)

# Published parameters
FICO_GAP_PTS   = 57.0   # Experian 2024
FICO_ELAS_BASE = 0.08   # Avery et al. 2006
DTI_GAP_BASE   = 4.0    # SCF 2022 conditional gap
DTI_GAP_HIGH   = 7.0    # Upper bound
DTI_ELAS_BASE  = 0.15   # GSE eligibility tables
DTI_ELAS_HIGH  = 0.20   # Upper bound

MaxFICO  = FICO_GAP_PTS  * FICO_ELAS_BASE   # 4.56 pp
MaxDTI_b = DTI_GAP_BASE  * DTI_ELAS_BASE    # 0.60 pp
MaxDTI_h = DTI_GAP_HIGH  * DTI_ELAS_HIGH    # 1.40 pp
MaxJ_b   = MaxFICO + MaxDTI_b               # 5.16 pp
MaxJ_h   = MaxFICO + MaxDTI_h               # 5.96 pp

print('Max FICO explanatory power:           {:.2f} pp'.format(MaxFICO))
print('Max DTI explanatory power (base):     {:.2f} pp'.format(MaxDTI_b))
print('Max DTI explanatory power (generous): {:.2f} pp'.format(MaxDTI_h))
print('Joint max (conservative):             {:.2f} pp'.format(MaxJ_b))
print('Joint max (generous):                 {:.2f} pp'.format(MaxJ_h))


In [ ]:
obs_gaps = {2020: 14.70, 2021: 13.01, 2022: 14.19, 2023: 15.36, 2024: 14.97}

dfl_dti_path = TABS / 'table_25_dfl_with_dti.csv'
dti_adj = {}
if dfl_dti_path.exists():
    df_dfl = pd.read_csv(dfl_dti_path)
    for yr in obs_gaps:
        sub = df_dfl[(df_dfl['Year']==yr) & (df_dfl['Spec']=='With DTI')]
        if len(sub):
            dti_adj[yr] = float(sub['Unexplained_pp'].values[0])
    print('DTI-adjusted gaps from NB25: {} years loaded'.format(len(dti_adj)))
else:
    print('Note: run NB25 first to add empirical DTI-adjusted gap column')

rows = []
for yr, obs in obs_gaps.items():
    lb_f  = max(0.0, obs - MaxFICO)
    lb_jb = max(0.0, obs - MaxJ_b)
    lb_jh = max(0.0, obs - MaxJ_h)
    row = {
        'Year': yr,
        'Obs_Gap_pp':        round(obs,  2),
        'MaxFICO_pp':        round(MaxFICO, 2),
        'MaxDTI_cons_pp':    round(MaxDTI_b, 2),
        'MaxDTI_gen_pp':     round(MaxDTI_h, 2),
        'LB_FICO_only_pp':   round(lb_f,  2),
        'LB_FICO_only_pct':  round(lb_f/obs*100,  1),
        'LB_Joint_cons_pp':  round(lb_jb, 2),
        'LB_Joint_cons_pct': round(lb_jb/obs*100, 1),
        'LB_Joint_gen_pp':   round(lb_jh, 2),
        'LB_Joint_gen_pct':  round(lb_jh/obs*100, 1),
    }
    if yr in dti_adj:
        row['DTI_Adjusted_pp'] = round(dti_adj[yr], 2)
    rows.append(row)
    print('  {}: Obs={:.2f}  FICO-LB={:.2f}({:.0f}%)  Joint-b-LB={:.2f}({:.0f}%)  Joint-h-LB={:.2f}({:.0f}%)'.format(
        yr, obs, lb_f, lb_f/obs*100, lb_jb, lb_jb/obs*100, lb_jh, lb_jh/obs*100))

df_bounds = pd.DataFrame(rows)
means = df_bounds.select_dtypes('number').mean()
mean_row = {'Year': 'Mean'}
mean_row.update({k: round(v,2) for k,v in means.items()})
df_bounds = pd.concat([df_bounds, pd.DataFrame([mean_row])], ignore_index=True)
df_bounds.to_csv(TABS / 'table_26_joint_manski_bounds.csv', index=False)

mean_obs   = np.mean(list(obs_gaps.values()))
mean_lb_f  = float(df_bounds.iloc[:-1]['LB_FICO_only_pp'].mean())
mean_lb_jb = float(df_bounds.iloc[:-1]['LB_Joint_cons_pp'].mean())
mean_lb_jh = float(df_bounds.iloc[:-1]['LB_Joint_gen_pp'].mean())

print()
print('Mean observed gap:                {:.2f} pp'.format(mean_obs))
print('Mean LB FICO only:                {:.2f} pp ({:.1f}%)'.format(mean_lb_f,  mean_lb_f/mean_obs*100))
print('Mean LB joint (conservative):     {:.2f} pp ({:.1f}%)'.format(mean_lb_jb, mean_lb_jb/mean_obs*100))
print('Mean LB joint (generous):         {:.2f} pp ({:.1f}%)'.format(mean_lb_jh, mean_lb_jh/mean_obs*100))
print('Saved: table_26_joint_manski_bounds.csv')


In [ ]:
yrs = [r['Year'] for r in rows]
obs_list   = [r['Obs_Gap_pp']       for r in rows]
lb_f_list  = [r['LB_FICO_only_pp']  for r in rows]
lb_jb_list = [r['LB_Joint_cons_pp'] for r in rows]
lb_jh_list = [r['LB_Joint_gen_pp']  for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(yrs, obs_list,   'k-o', lw=2, ms=8, label='Observed gap', zorder=5)
lbl1 = 'FICO max explained ({:.2f}pp)'.format(MaxFICO)
lbl2 = 'DTI max explained (conservative)'
lbl3 = 'DTI max explained (generous add-on)'
lbl4 = 'Joint LB (conservative): mean {:.2f}pp'.format(mean_lb_jb)
lbl5 = 'Joint LB (generous): mean {:.2f}pp'.format(mean_lb_jh)
ax.fill_between(yrs, lb_f_list,  obs_list,   alpha=0.20, color='#E53935', label=lbl1)
ax.fill_between(yrs, lb_jb_list, lb_f_list,  alpha=0.30, color='#FB8C00', label=lbl2)
ax.fill_between(yrs, lb_jh_list, lb_jb_list, alpha=0.20, color='#FFD54F', label=lbl3)
ax.plot(yrs, lb_jb_list, 'b--o', lw=2, ms=5, label=lbl4)
ax.plot(yrs, lb_jh_list, 'g:^',  lw=2, ms=5, label=lbl5)
ax.set_xlabel('Year'); ax.set_ylabel('Racial Approval Gap (pp)')
ax.set_title('Panel A: Joint FICO + DTI Manski Bounds', fontweight='bold')
ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=0.3)

ax = axes[1]
lbs = [mean_lb_f, mean_lb_jb, mean_lb_jh]
exps = [mean_obs - lb for lb in lbs]
clrs_lb = ['#1565C0', '#1976D2', '#42A5F5']
clrs_ex = ['#EF9A9A', '#FFCC80', '#FFF9C4']
cats = ['FICO only', 'Joint FICO+DTI\n(conservative)', 'Joint FICO+DTI\n(generous)']
for i, (lb, ex, clb, cex) in enumerate(zip(lbs, exps, clrs_lb, clrs_ex)):
    ax.bar(i, lb,  color=clb, alpha=0.90)
    ax.bar(i, ex, bottom=lb, color=cex, alpha=0.85, edgecolor='grey', linewidth=0.5)
    pct = lb / mean_obs * 100
    ax.text(i, lb/2, '{:.1f}pp\n({:.0f}%)'.format(lb, pct),
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(i, lb+ex/2, '{:.1f}pp'.format(ex), ha='center', va='center', fontsize=9)
ax.set_xticks(range(3)); ax.set_xticklabels(cats, fontsize=9)
ax.set_ylabel('Gap (pp)')
ax.set_title('Panel B: Mean Decomposition\nMean observed = {:.2f}pp'.format(mean_obs), fontweight='bold')
ax.grid(alpha=0.3, axis='y')

fig.suptitle('Figure 26: Joint FICO + DTI Manski Partial Identification Bounds',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / 'figure_26_joint_bounds.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figure_26_joint_bounds.png')
print('NB26 COMPLETE')
